In [ ]:
import pandas as pd

apps_with_duplicates = pd.read_csv('datasets/apps.csv')

apps = apps_with_duplicates.drop_duplicates()

print('Total number of apps in the dataset = ', len(apps))

print(apps.sample(5))


In [ ]:
chars_to_remove = ['+', ',', '$']
# Columns to clean
cols_to_clean = ['Installs', 'Price']

for col in cols_to_clean:
    for char in chars_to_remove:
        apps[col] = apps[col].apply(lambda x: str(x).replace(char, ''))

print(apps.info())


In [ ]:
import numpy as np

apps['Installs'] = apps['Installs'].astype(float)
apps['Price'] = apps['Price'].astype(float)

print(apps.dtypes)


In [ ]:
import plotly
plotly.offline.init_notebook_mode(connected=True)
import plotly.graph_objs as go

num_categories = len(apps['Category'].unique())
print('Number of categories = ', num_categories)

num_apps_in_category = apps['Category'].value_counts()

sorted_num_apps_in_category = num_apps_in_category.sort_values(ascending=False)

data = [go.Bar(
    x=sorted_num_apps_in_category.index,
    y=sorted_num_apps_in_category.values
)]

plotly.offline.iplot(data)


In [ ]:
# Average rating
avg_app_rating = apps['Rating'].mean()
print('Average app rating = ', avg_app_rating)

# Histogram of ratings
data = [go.Histogram(x=apps['Rating'])]

layout = {
    'shapes': [{
        'type': 'line',
        'x0': avg_app_rating,
        'y0': 0,
        'x1': avg_app_rating,
        'y1': 1000,
        'line': {'dash': 'dashdot'}
    }]
}

plotly.offline.iplot({'data': data, 'layout': layout})


In [ ]:
%matplotlib inline
import seaborn as sns
sns.set_style("darkgrid")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Apps with both Size and Rating available
apps_with_size_and_rating_present = apps[apps['Size'].notnull() & apps['Rating'].notnull()]

# Subset for categories with at least 250 apps
large_categories = apps_with_size_and_rating_present.groupby('Category').filter(lambda x: len(x) >= 250)

# Size vs Rating
plt1 = sns.jointplot(x=large_categories['Size'], y=large_categories['Rating'])

# Paid apps only
paid_apps = apps[apps['Type'] == 'Paid']

# Price vs Rating
plt2 = sns.jointplot(x=paid_apps['Price'], y=paid_apps['Rating'])


In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(15, 8)

# Popular categories
popular_app_cats = apps[apps.Category.isin([
    'GAME', 'FAMILY', 'PHOTOGRAPHY',
    'MEDICAL', 'TOOLS', 'FINANCE',
    'LIFESTYLE', 'BUSINESS'
])]

# Plot price vs category
ax = sns.stripplot(x=popular_app_cats['Category'], y=popular_app_cats['Price'], jitter=True, linewidth=1)
ax.set_title('App pricing trend across categories')

# Apps with price > 200
apps_above_200 = apps[apps['Price'] > 200]
apps_above_200[['Category', 'App', 'Price']]


In [ ]:
# Apps priced below $100
apps_under_100 = apps[apps['Price'] < 100]

fig, ax = plt.subplots()
fig.set_size_inches(15, 8)

# Price vs category after filtering
ax = sns.stripplot(x='Category', y='Price', data=apps_under_100, jitter=True, linewidth=1)
ax.set_title('App pricing trend across categories after filtering for junk apps')


In [ ]:
trace0 = go.Box(
    y=apps[apps['Type'] == 'Paid']['Installs'],
    name='Paid'
)

trace1 = go.Box(
    y=apps[apps['Type'] == 'Free']['Installs'],
    name='Free'
)

layout = go.Layout(
    title="Number of downloads of paid apps vs. free apps",
    yaxis=dict(title="Log number of downloads", type='log', autorange=True)
)

data = [trace0, trace1]
plotly.offline.iplot({'data': data, 'layout': layout})


In [ ]:
# Load user reviews
reviews_df = pd.read_csv('datasets/user_reviews.csv')

# Join on App name
merged_df = pd.merge(apps, reviews_df, on='App', how='inner')

# Drop missing Sentiment or Review
merged_df = merged_df.dropna(subset=['Sentiment', 'Review'])

sns.set_style('ticks')
fig, ax = plt.subplots()
fig.set_size_inches(11, 8)

# Sentiment polarity boxplot
ax = sns.boxplot(x='Type', y='Sentiment_Polarity', data=merged_df)
ax.set_title('Sentiment Polarity Distribution')
